# AI Playwright Test Generator — Pipeline Debugger

Run cells **top to bottom** for a full pipeline trace.  
Re-run any individual cell after editing config or patching a module — state is preserved between cells.

| Cell | Stage | What to look for |
|------|-------|------------------|
| 1 | Setup | No import errors, kernel shows project venv |
| 2 | Config | Correct URL and story before proceeding |
| 3 | LLM Skeleton | Raw `{{CLICK:...}}` tokens present, no guessed selectors |
| 4 | Scraper | Candidate count per page, element types found |
| 4b | Search | Filter candidates by token description |
| 5 | Resolution | Per-token status, winner selector, scores, fallback flags |
| 5b | Deep-dive | Top 10 candidates for a failing token |
| 6 | Output | Final test file, `pytest.skip()` count |
| 7 | Patch | Manual selector override without code changes |

In [1]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
# Adds src/ to path and patches the event loop so sync Playwright
# doesn't clash with Jupyter's own asyncio loop.

import sys
from pathlib import Path

# Resolve project root (one level up from /notebooks)
PROJECT_ROOT = Path.cwd()
if "notebooks" in PROJECT_ROOT.parts:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Patch asyncio BEFORE importing anything that touches Playwright
import nest_asyncio
nest_asyncio.apply()

import pandas as pd
from IPython.display import display

# Core pipeline modules
from src.llm_client import LLMClient
from src.orchestrator import TestOrchestrator
from src.scraper import PageScraper
from src.semantic_candidate_ranker import SemanticCandidateRanker
from src.placeholder_resolver import PlaceholderResolver
from src.placeholder_orchestrator import PlaceholderOrchestrator
from src.skeleton_parser import SkeletonParser
from src.test_generator import TestGenerator

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")
print("All imports OK")

Project root: c:\Users\l_a_c\code\AI-Playwright-Test-Generator
Python: 3.13.7
All imports OK


In [2]:
# ── Cell 2: Config ────────────────────────────────────────────────────────────
# Edit these values before running the rest of the notebook.

TARGET_URL = "https://automationexercise.com"

USER_STORY = """
As a customer, I want to browse products and add them to my cart
"""

CONDITIONS = """
1. Navigate to the automationexercise.com home page
2. Click the 'Products' link in the header navigation
3. Click 'Add to cart' for a product
4. Verify confirmation message appears
"""

# LLM provider config — use LM Studio to avoid GPU VRAM contention
# when Cline is already running with a model loaded.
LLM_PROVIDER = "lm-studio"
MODEL_NAME = None  # Auto-detect from provider

print(f"Target URL    : {TARGET_URL}")
print(f"LLM Provider  : {LLM_PROVIDER}")
print(f"Model         : {MODEL_NAME or 'auto-detect'}")

Target URL    : https://automationexercise.com
LLM Provider  : lm-studio
Model         : auto-detect


In [3]:
# ── Cell 3: LLM Skeleton Generation ───────────────────────────────────────────
# What to look for:
#   ✅ Every action step has a {{{{TOKEN:description}}}} — no raw CSS or XPath
#   ❌ If you see page.locator('#some-id') the LLM guessed — check your prompt

import re

# Configure the LLM provider
LLMClient.set_session_provider(provider=LLM_PROVIDER, model=MODEL_NAME)
client = LLMClient()
print(f"Using: provider={client.provider_name}, model={client.model}")

generator = TestGenerator(client=client)
raw_skeleton = await generator.generate_skeleton(
    user_story=USER_STORY,
    conditions=CONDITIONS,
    target_urls=[TARGET_URL],
    expected_count=6,
)

# Normalize single-brace {ACTION:desc} → double-brace {{ACTION:desc}}
# LLMs trained on Python f-strings emit single braces when prompted with double braces.
skeleton = SkeletonParser.normalise_placeholder_actions(raw_skeleton)

# Token inventory
TOKEN_RE = re.compile(r"\{\{(\w+):([^}]+)\}\}")
tokens = TOKEN_RE.findall(skeleton)

print(f"\nTokens found: {len(tokens)}")
print("-" * 60)
for kind, desc in tokens:
    print(f"  [{kind:10s}] {desc}")

# Unique tokens we'll test resolution against
unique_tokens = sorted(set((kind, desc.strip()) for kind, desc in tokens))
print(f"\nUnique placeholders: {len(unique_tokens)}")
for kind, desc in unique_tokens:
    print(f"  [{kind:10s}] {desc}")

print("\n" + "=" * 60 + "\nRAW SKELETON\n" + "=" * 60)
print(skeleton)

[llm_client] Using loaded model: unsloth/qwen3.6-27b
Using: provider=lm-studio, model=unsloth/qwen3.6-27b
[llm_client] Calling provider=lm-studio model=unsloth/qwen3.6-27b timeout=600
[llm_client] Received completion in 409.86s, length=1424 chars

Tokens found: 21
------------------------------------------------------------
  [GOTO      ] home
  [ASSERT    ] home page header
  [GOTO      ] home
  [CLICK     ] products link
  [ASSERT    ] products page loaded
  [GOTO      ] home
  [CLICK     ] products link
  [CLICK     ] add to cart button
  [ASSERT    ] product added confirmation
  [GOTO      ] home
  [CLICK     ] products link
  [CLICK     ] add to cart button
  [ASSERT    ] confirmation message appears
  [GOTO      ] home
  [CLICK     ] products link
  [CLICK     ] add to cart button for second product
  [ASSERT    ] confirmation message appears
  [GOTO      ] home
  [CLICK     ] products link
  [CLICK     ] add to cart button
  [ASSERT    ] cart badge updated

Unique placeholders: 

In [4]:
# ── Cell 4: Scraper Run ───────────────────────────────────────────────────────
# What to look for:
#   ✅ Candidates > 0 for the pages your tokens reference
#   ❌ Empty candidate list usually means auth wall or wrong URL

import asyncio

scraper = PageScraper()
raw_result = await scraper.scrape_url(TARGET_URL)
candidates, error, final_url = raw_result

if error:
    print(f"Scrape error: {error}")
else:
    print(f"Scraped URL: {final_url}")

# Build a readable DataFrame
rows = []
for elem in candidates:
    rows.append(
        {
            "tag": elem.get("tag", ""),
            "text": (elem.get("text") or "")[:60],
            "role": elem.get("role", ""),
            "id": elem.get("id", ""),
            "name": elem.get("name", ""),
            "placeholder": elem.get("placeholder", ""),
            "selector": (elem.get("selector") or "")[:80],
            "is_visible": elem.get("is_visible", "?"),
        }
    )

df = pd.DataFrame(rows)
print(f"\nTotal candidates scraped: {len(df)}")
if len(df) > 0:
    print(f"Tag breakdown:\n{df['tag'].value_counts().to_string()}")
    display(df)

[scraper] Starting browser scrape for https://automationexercise.com...
[scraper] CDP accessibility tree captured: 3946 nodes



Scraped URL: https://automationexercise.com/

Total candidates scraped: 1071
Tag breakdown:
tag
    1071


,tag,text,role,id,name,placeholder,selector,is_visible
0,,,a,,,,"a[href=""/""]",True
1,,Home,a,,,,"a[href=""/""]",True
2,, Products,a,,,,"a[href=""/products""]",True
3,,Cart,a,,,,"a[href=""/view_cart""]",True
4,,Signup / Login,a,,,,"a[href=""/login""]",True
...,...,...,...,...,...,...,...,...
1066,,,button,,,,.fc-preference-consent.atp-vendor,False
1067,,Accept all,button,,,,.fc-button.fc-vendor-preferences-accept-all.fc...,False
1068,,Confirm choices,button,,,,.fc-button.fc-confirm-choices.fc-primary-button,False
1069,,Close,button,,,,.fc-help-dialog-close-button,False


In [5]:
# ── Cell 4b: Search Candidates by Token Description ──────────────────────────
# Filter the scraped candidates to find elements matching a specific token.
# Change SEARCH_TOKEN to match any token from Cell 3.

SEARCH_TOKEN = "Products"  # Change this to search different tokens

def search_candidates(df, query):
    """Search candidates across text, role, id, name, placeholder fields."""
    query_lower = query.lower()
    words = query_lower.split()
    
    results = []
    for idx, row in df.iterrows():
        searchable = " ".join([
            str(row.get("text", "")),
            str(row.get("role", "")),
            str(row.get("id", "")),
            str(row.get("name", "")),
            str(row.get("placeholder", "")),
        ]).lower()
        
        # Count how many query words match
        matches = sum(1 for w in words if w in searchable and len(w) > 2)
        if matches > 0:
            results.append((matches, idx, row))
    
    results.sort(key=lambda x: x[0], reverse=True)
    return results

if "df" in globals() and len(df) > 0:
    results = search_candidates(df, SEARCH_TOKEN)
    print(f"Candidates matching '{SEARCH_TOKEN}': {len(results)}")
    
    if results:
        match_rows = []
        for score, idx, row in results[:15]:
            match_rows.append({"match_score": score, **row.to_dict()})
        match_df = pd.DataFrame(match_rows)
        display(match_df)
    else:
        print("  No matching candidates found on this page.")
        print("  This element may be on a different page (e.g., after login).")
else:
    print("No scraper data available. Run Cell 4 first.")

Candidates matching 'Products': 1


,match_score,tag,text,role,id,name,placeholder,selector,is_visible
0,1,, Products,a,,,,"a[href=""/products""]",True


In [7]:
# ── Cell 5: Placeholder Resolution ────────────────────────────────────────────
# Resolve each unique placeholder against the scraped candidates.
# What to look for:
#   - Resolved: element found with good score
#   - Skipped: no matching element (may need different page scraped)

ranker = SemanticCandidateRanker(None)
resolver = PlaceholderResolver()

# Build per-token resolution results
report_rows = []
for action, desc in unique_tokens:
    scored = resolver.rank_candidates(action, desc, candidates)
    
    if scored:
        top_score, top_elem = scored[0]
        selector = (top_elem.get("selector") or "")[:60]
        
        # Check text validation
        text_valid = resolver.text_matches_description(
            top_elem.get("text", ""), desc
        ) if top_elem.get("text") else True
        
        status = "resolved" if text_valid else "text_mismatch"
        report_rows.append({
            "token": f"{{{action}:{desc}}}",
            "status": status,
            "winner": selector,
            "score": round(top_score, 2),
            "candidates": len(scored),
            "text_validated": text_valid,
        })
    else:
        report_rows.append({
            "token": f"{{{action}:{desc}}}",
            "status": "no_candidates",
            "winner": "",
            "score": None,
            "candidates": 0,
            "text_validated": False,
        })

report_df = pd.DataFrame(report_rows)
resolved_count = len(report_df[report_df["status"] == "resolved"])
skipped_count = len(report_df[report_df["status"] != "resolved"])

print(f"Resolved : {resolved_count} / {len(report_df)}")
print(f"Skipped  : {skipped_count} / {len(report_df)}")
print()

# Color-coded status column
def style_status(v):
    if v == "resolved":
        return "background-color: #d4edda"
    elif v == "text_mismatch":
        return "background-color: #fff3cd"
    elif v == "no_candidates":
        return "background-color: #f8d7da"
    return ""

display(report_df.style.map(style_status, subset=["status"]))

Resolved : 6 / 9
Skipped  : 3 / 9



,token,status,winner,score,candidates,text_validated
0,{ASSERT:cart badge updated},resolved,"a.btn.btn-default.add-to-cart[data-product-id=""11""]",8.000000,74,True
1,{ASSERT:confirmation message appears},text_mismatch,.btn.btn-success,2.000000,87,False
2,{ASSERT:home page header},no_candidates,,nan,0,False
3,{ASSERT:product added confirmation},resolved,"a[href=""/product_details/11""]",5.000000,121,True
4,{ASSERT:products page loaded},no_candidates,,nan,0,False
5,{CLICK:add to cart button},resolved,"a.btn.btn-default.add-to-cart[data-product-id=""11""]",22.000000,74,True
6,{CLICK:add to cart button for second product},resolved,"a.btn.btn-default.add-to-cart[data-product-id=""11""]",22.000000,74,True
7,{CLICK:products link},resolved,"a[href=""/products""]",13.000000,143,True
8,{GOTO:home},resolved,"a[href=""/""]",100.000000,1,True


In [8]:
# ── Cell 5b: Deep-dive a single failing token ─────────────────────────────────
# Paste the token description from the skipped rows above.
# Shows every candidate the ranker saw and its raw score.

FAILING_TOKEN = "'Products' link in header navigation"
ACTION_FOR_TOKEN = "CLICK"

scored = resolver.rank_candidates(ACTION_FOR_TOKEN, FAILING_TOKEN, candidates)

scored_df = pd.DataFrame(
    [
        {
            "score": round(s[0], 2),
            "tag": s[1].get("tag", ""),
            "text": (s[1].get("text") or "")[:60],
            "role": s[1].get("role", ""),
            "id": s[1].get("id", ""),
            "name": s[1].get("name", ""),
            "selector": (s[1].get("selector") or "")[:80],
            "is_visible": s[1].get("is_visible", "?"),
        }
        for s in scored
    ]
).sort_values("score", ascending=False)

def vis_style(v):
    if v is False:
        return "background-color: #f8d7da"
    elif v is True:
        return "background-color: #d4edda"
    return ""

print(f"Top 10 candidates for '{FAILING_TOKEN}':")
if len(scored_df) > 0:
    display(scored_df.head(10).style.map(vis_style, subset=["is_visible"]))
else:
    print("  No candidates found on this page.")
    print("  This element is likely on a page requiring login (inventory.html).")
    print("  The journey scraper should capture it after login.")

Top 10 candidates for ''Products' link in header navigation':


,score,tag,text,role,id,name,selector,is_visible
0,8,,(3) Allen Solly Junior,a,,,"a[href=""/brand_products/Allen Solly Junior""]",True
1,8,,(3) Mast & Harbour,a,,,"a[href=""/brand_products/Mast & Harbour""]",True
2,8,,(3) Kookie Kids,a,,,"a[href=""/brand_products/Kookie Kids""]",True
3,8,,(4) Babyhug,a,,,"a[href=""/brand_products/Babyhug""]",True
4,8,,(5) Madame,a,,,"a[href=""/brand_products/Madame""]",True
5,8,, Products,a,,,"a[href=""/products""]",True
6,8,,(6) Polo,a,,,"a[href=""/brand_products/Polo""]",True
7,8,,(5) Biba,a,,,"a[href=""/brand_products/Biba""]",True
8,8,,(5) H&M,a,,,"a[href=""/brand_products/H&M""]",True
9,6,,APIs list for practice,a,,,"a[href=""/api_list""]",True


In [9]:
# ── Cell 6: Full Pipeline Run (optional) ──────────────────────────────────────
# Run the complete orchestrator pipeline to see the full end-to-end result.
# This combines skeleton generation + scraping + placeholder resolution.

import time

orchestrator = TestOrchestrator(generator)

start = time.time()
final_code = await orchestrator.run_pipeline(
    user_story=USER_STORY,
    conditions=CONDITIONS,
    target_urls=[TARGET_URL],
)
duration = time.time() - start

skip_count = final_code.count("pytest.skip")
test_count = len(re.findall(r"def test_\d+", final_code))

print(f"Pipeline complete in {duration:.1f}s")
print(f"Tests generated: {test_count}")
print(f"pytest.skip() calls: {skip_count}")
print("=" * 60)
print("FINAL TEST FILE")
print("=" * 60)
print(final_code)

[pipeline] phase=generate_skeleton start
[llm_client] Calling provider=lm-studio model=unsloth/qwen3.6-27b timeout=600
[llm_client] Received completion in 306.27s, length=1021 chars
[pipeline] phase=generate_skeleton done
[pipeline] phase=scrape start urls=1


[scraper] Starting browser scrape for https://automationexercise.com...
[scraper] CDP accessibility tree captured: 3946 nodes



[pipeline] phase=journey_discovery start
[pipeline] following discovery journey for: test_01_navigate_to_home


[journey_discovery] Navigating to starting URL: https://automationexercise.com
[journey_discovery] Step 1/1: scrape 'home page loaded'



[pipeline] following discovery journey for: test_02_click_products_link


...

[pipeline] following discovery journey for: test_03_click_add_to_cart


[journey_discovery] Navigating to starting URL: https://automationexercise.com
[journey_discovery] Step 1/3: click 'products link in header navigation'
[journey_discovery] Scraped 156 elements for discovery of 'products link in header navigation'
[journey_discovery] Attempting to click selector: a[href="/brand_products/Allen Solly Junior"]
[journey_discovery] Clicked successfully: a[href="/brand_products/Allen Solly Junior"]
[journey_discovery] Step 2/3: click 'add to cart button for a product'
[journey_discovery] Scraped 43 elements for discovery of 'add to cart button for a product'
[journey_discovery] Attempting to click selector: .add-to-cart.btn[data-product-id="13"]
[journey_discovery] Clicked successfully: .add-to-cart.btn[data-product-id="13"]
[journey_discovery] Step 3/3: scrape 'add to cart button clicked'



[pipeline] following discovery journey for: test_04_verify_confirmation_message


[journey_discovery] Navigating to starting URL: https://automationexercise.com
[journey_discovery] Step 1/3: click 'products link in header navigation'
[journey_discovery] Scraped 156 elements for discovery of 'products link in header navigation'
[journey_discovery] Attempting to click selector: a[href="/brand_products/Allen Solly Junior"]
[journey_discovery] Clicked successfully: a[href="/brand_products/Allen Solly Junior"]
[journey_discovery] Step 2/3: click 'add to cart button for a product'
[journey_discovery] Scraped 43 elements for discovery of 'add to cart button for a product'
[journey_discovery] Attempting to click selector: .add-to-cart.btn[data-product-id="13"]
[journey_discovery] Clicked successfully: .add-to-cart.btn[data-product-id="13"]
[journey_discovery] Step 3/3: scrape 'confirmation message appears'



[pipeline] journey discovery enriched: https://automationexercise.com (156 elements)
[pipeline] journey discovery enriched: https://automationexercise.com/brand_products/Allen%20Solly%20Junior (43 elements)
[pipeline] phase=journey_discovery done
[pipeline] phase=scrape done
[pipeline] url_resolver mappings: {'home': 'https://automationexercise.com', 'login': 'https://automationexercise.com', 'homepage': 'https://automationexercise.com'}
[pipeline] phase=resolve_placeholders start
[DEBUG] Failed to find 'products link in header navigation'. Available scraped URLs: ['https://automationexercise.com', 'https://automationexercise.com/brand_products/Allen%20Solly%20Junior']
[DEBUG] Failed to find 'products link in header navigation'. Available scraped URLs: ['https://automationexercise.com', 'https://automationexercise.com/brand_products/Allen%20Solly%20Junior']
[DEBUG] Failed to find 'products link in header navigation'. Available scraped URLs: ['https://automationexercise.com', 'https://a

In [ ]:
# ── Cell 7: Patch and re-test a selector manually ────────────────────────────
# When cell 5b shows the right element exists but didn't win,
# use this cell to inject a manual override and re-run resolution
# without restarting the whole pipeline.
#
# This does NOT write back to the codebase - it's a scratchpad.

OVERRIDE_TOKEN = "add to cart button for Sauce Labs Backpack"
OVERRIDE_SELECTOR = "#add-to-cart-from-qty"

patched = final_code.replace(
    'pytest.skip("Skipping: unresolved placeholders for:',
    '# ORIGINAL: pytest.skip("Skipping: unresolved placeholders for:',
)
patched = patched.replace(
    f"{{{{CLICK:{OVERRIDE_TOKEN}}}}}}",
    f'page.locator("{OVERRIDE_SELECTOR}").click()  # manually patched'
)

print("Patched code (first 2000 chars):")
print(patched[:2000])
print("...\n")
print("(Full patched code available in the 'patched' variable)")